# 🌱 Smart Crop Advisor — Fertilizer Recommendation Model Training

> **Module:** AI/ML — Fertilizer Prediction  
> **Algorithm:** Random Forest Classifier  
> **Output:** `../models/fertilizer_model.pkl`

This notebook covers the full ML pipeline:
1. Load & Explore Dataset (EDA)
2. Preprocess Data (Encoding + Scaling)
3. Train/Test Split
4. Train RandomForest with GridSearchCV
5. Evaluate Model
6. Save Model Artifacts
7. Sanity-check Inference

---
## 📦 Step 0 — Import Libraries

In [16]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe for script execution
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble         import RandomForestClassifier
from sklearn.model_selection  import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.metrics          import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import sklearn
print('✅ All libraries imported successfully')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   sklearn : {sklearn.__version__}')
print(f'   joblib  : {joblib.__version__}')

✅ All libraries imported successfully
   pandas  : 3.0.3
   numpy   : 2.4.6
   sklearn : 1.9.0
   joblib  : 1.5.3


---
## 📂 Step 1 — Load Dataset

In [17]:
# Resolve paths relative to the notebook location
NOTEBOOK_DIR = os.path.abspath('')          # ai/notebooks/
AI_DIR       = os.path.dirname(NOTEBOOK_DIR)  # ai/
DATA_DIR     = os.path.join(AI_DIR, 'data')
MODELS_DIR   = os.path.join(AI_DIR, 'models')

os.makedirs(MODELS_DIR, exist_ok=True)

print(f'DATA_DIR   : {DATA_DIR}')
print(f'MODELS_DIR : {MODELS_DIR}')
print(f'Files in data/ : {os.listdir(DATA_DIR)}')

DATA_DIR   : c:\Users\Arpit Gupta\OneDrive\Desktop\SmartCrop\Smart-Crop-Advisor\ai\data
MODELS_DIR : c:\Users\Arpit Gupta\OneDrive\Desktop\SmartCrop\Smart-Crop-Advisor\ai\models
Files in data/ : ['Crop_recommendation.csv', 'Fertilizer Prediction.csv']


In [18]:
# Auto-detect the fertilizer CSV
csv_file = None
for f in os.listdir(DATA_DIR):
    if 'fertil' in f.lower() and f.lower().endswith('.csv'):
        csv_file = os.path.join(DATA_DIR, f)
        break

if csv_file is None:
    raise FileNotFoundError('No fertilizer CSV found in data/ folder!')

print(f'📄 Loading: {csv_file}')

# Read CSV — force all string-like columns to stay as plain Python str (object dtype)
# This is important for pandas 3.x which defaults to ArrowDtype / StringDtype
df = pd.read_csv(csv_file, dtype_backend='numpy_nullable')

# Strip whitespace from column names & string values
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Convert nullable integer / boolean columns to plain numpy types
df = df.convert_dtypes(convert_string=False)   # keep strings as object, cast Int64→int64

print(f'✅ Dataset loaded — Shape: {df.shape}')
df.head(10)

📄 Loading: c:\Users\Arpit Gupta\OneDrive\Desktop\SmartCrop\Smart-Crop-Advisor\ai\data\Fertilizer Prediction.csv
✅ Dataset loaded — Shape: (99, 9)


,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,Sandy,Maize,37,0,0,Urea
1,29,52,45,Loamy,Sugarcane,12,0,36,DAP
2,34,65,62,Black,Cotton,7,9,30,14-35-14
3,32,62,34,Red,Tobacco,22,0,20,28-28
4,28,54,46,Clayey,Paddy,35,0,0,Urea
5,26,52,35,Sandy,Barley,12,10,13,17-17-17
6,25,50,64,Red,Cotton,9,0,10,20-20
7,33,64,50,Loamy,Wheat,41,0,0,Urea
8,30,60,42,Sandy,Millets,21,0,18,28-28
9,29,58,33,Black,Oil seeds,9,7,30,14-35-14


---
## 🔍 Step 2 — Exploratory Data Analysis (EDA)

In [19]:
print('=' * 55)
print('  DATASET OVERVIEW')
print('=' * 55)
print(f'  Rows    : {df.shape[0]}')
print(f'  Columns : {df.shape[1]}')
print(f'  Names   : {list(df.columns)}')
print('\n📋 Data Types:')
print(df.dtypes)
print('\n🔎 Missing Values:')
print(df.isnull().sum())
print('\n📊 Duplicates:', df.duplicated().sum())

  DATASET OVERVIEW
  Rows    : 99
  Columns : 9
  Names   : ['Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type', 'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer Name']

📋 Data Types:
Temparature         Int64
Humidity            Int64
Moisture            Int64
Soil Type          string
Crop Type          string
Nitrogen            Int64
Potassium           Int64
Phosphorous         Int64
Fertilizer Name    string
dtype: object

🔎 Missing Values:
Temparature        0
Humidity           0
Moisture           0
Soil Type          0
Crop Type          0
Nitrogen           0
Potassium          0
Phosphorous        0
Fertilizer Name    0
dtype: int64

📊 Duplicates: 0


In [20]:
print('\n📈 Statistical Summary:')
df.describe(include='all')


📈 Statistical Summary:


,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
count,99.0,99.0,99.0,99,99,99.0,99.0,99.0,99
unique,<NA>,<NA>,<NA>,5,11,<NA>,<NA>,<NA>,7
top,<NA>,<NA>,<NA>,Loamy,Sugarcane,<NA>,<NA>,<NA>,Urea
freq,<NA>,<NA>,<NA>,21,13,<NA>,<NA>,<NA>,22
mean,30.282828,59.151515,43.181818,NaN,NaN,18.909091,3.383838,18.606061,NaN
std,3.502304,5.840331,11.271568,NaN,NaN,11.599693,5.814667,13.476978,NaN
min,25.0,50.0,25.0,NaN,NaN,4.0,0.0,0.0,NaN
25%,28.0,54.0,34.0,NaN,NaN,10.0,0.0,9.0,NaN
50%,30.0,60.0,41.0,NaN,NaN,13.0,0.0,19.0,NaN
75%,33.0,64.0,50.5,NaN,NaN,24.0,7.5,30.0,NaN


In [21]:
# ── Identify Target column ─────────────────────────────────────────────────
TARGET_COL = None
for col in df.columns:
    if 'fertilizer' in col.lower():
        TARGET_COL = col
        break
if TARGET_COL is None:
    TARGET_COL = df.columns[-1]       # fallback: last column

# ── Identify Feature columns ────────────────────────────────────────────────
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]

# ── Detect Numeric vs Categorical using pandas API (works in pandas 1-3) ────
#    pd.api.types.is_numeric_dtype handles int64, float64, Int64, Float64, etc.
NUM_COLS = [c for c in FEATURE_COLS if pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = [c for c in FEATURE_COLS if not pd.api.types.is_numeric_dtype(df[c])]

print(f'🎯 Target Column     : "{TARGET_COL}"')
print(f'🔢 Numeric features  : {NUM_COLS}')
print(f'🔤 Categorical features: {CAT_COLS}')
print(f'\n🌿 Unique Fertilizer classes ({df[TARGET_COL].nunique()}):')
print(df[TARGET_COL].value_counts().to_string())

🎯 Target Column     : "Fertilizer Name"
🔢 Numeric features  : ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
🔤 Categorical features: ['Soil Type', 'Crop Type']

🌿 Unique Fertilizer classes (7):
Fertilizer Name
Urea        22
DAP         18
28-28       17
14-35-14    14
20-20       14
17-17-17     7
10-26-26     7


In [22]:
# Fertilizer class distribution bar chart
vc = df[TARGET_COL].value_counts()
fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(vc)))
vc.plot(kind='bar', ax=ax, color=colors, edgecolor='black', width=0.7)
ax.set_title('Fertilizer Class Distribution', fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Fertilizer Name', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.tick_params(axis='x', rotation=30)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'fertilizer_class_dist.png'), dpi=100)
plt.show()
print('✅ Class distribution saved')

✅ Class distribution saved


In [23]:
# Numeric feature distributions
if NUM_COLS:
    cols_per_row = 3
    rows = (len(NUM_COLS) + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, 4 * rows))
    axes = np.array(axes).flatten()
    for i, col in enumerate(NUM_COLS):
        axes[i].hist(df[col].astype(float), bins=20,
                     color='steelblue', edgecolor='white', alpha=0.85)
        axes[i].set_title(f'{col} Distribution', fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')
    for j in range(len(NUM_COLS), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR, 'fertilizer_numeric_dist.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print('✅ Numeric distributions saved')

✅ Numeric distributions saved


In [24]:
# Correlation heatmap
if len(NUM_COLS) >= 2:
    corr = df[NUM_COLS].astype(float).corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                mask=mask, ax=ax, linewidths=0.5, square=True)
    ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold', pad=14)
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR, 'fertilizer_correlation.png'), dpi=100)
    plt.show()
    print('✅ Correlation heatmap saved')

✅ Correlation heatmap saved


---
## ⚙️ Step 3 — Preprocessing

**Fix for pandas 3.x:** We use `pd.api.types.is_numeric_dtype()` for column type detection
and convert all data to a plain numpy float64 array before fitting the model.

In [25]:
df_proc = df.copy()

# ── Encode categorical FEATURE columns → integers ──────────────────────────
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col].astype(str).str.strip())
    label_encoders[col] = le
    print(f'  ✔ "{col}" encoded  →  classes: {list(le.classes_)}')

# ── Encode TARGET column ───────────────────────────────────────────────────
target_encoder = LabelEncoder()
df_proc[TARGET_COL] = target_encoder.fit_transform(
    df_proc[TARGET_COL].astype(str).str.strip()
)
print(f'\n  ✔ Target "{TARGET_COL}" encoded  →  classes: {list(target_encoder.classes_)}')

  ✔ "Soil Type" encoded  →  classes: ['Black', 'Clayey', 'Loamy', 'Red', 'Sandy']
  ✔ "Crop Type" encoded  →  classes: ['Barley', 'Cotton', 'Ground Nuts', 'Maize', 'Millets', 'Oil seeds', 'Paddy', 'Pulses', 'Sugarcane', 'Tobacco', 'Wheat']

  ✔ Target "Fertilizer Name" encoded  →  classes: ['10-26-26', '14-35-14', '17-17-17', '20-20', '28-28', 'DAP', 'Urea']


In [26]:
# Separate X and y
# Convert to plain numpy float64 array — avoids any dtype issues in sklearn 1.x
X_raw = df_proc[FEATURE_COLS].astype(float).to_numpy(dtype=np.float64)
y     = df_proc[TARGET_COL].to_numpy(dtype=np.int64)

print(f'X shape : {X_raw.shape}  |  dtype: {X_raw.dtype}')
print(f'y shape : {y.shape}  |  dtype: {y.dtype}')
print(f'Classes : {list(target_encoder.classes_)}')

X shape : (99, 8)  |  dtype: float64
y shape : (99,)  |  dtype: int64
Classes : ['10-26-26', '14-35-14', '17-17-17', '20-20', '28-28', 'DAP', 'Urea']


In [27]:
# Scale ALL numeric feature columns using StandardScaler
# (Categorical columns are already encoded as small integers — also fine to scale)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)   # returns plain numpy ndarray

print(f'✅ StandardScaler applied')
print(f'   X_scaled shape : {X_scaled.shape}  |  dtype: {X_scaled.dtype}')
print(f'   Mean (first 3 cols) : {X_scaled[:, :3].mean(axis=0).round(4)}')
print(f'   Std  (first 3 cols) : {X_scaled[:, :3].std(axis=0).round(4)}')

✅ StandardScaler applied
   X_scaled shape : (99, 8)  |  dtype: float64
   Mean (first 3 cols) : [-0.  0.  0.]
   Std  (first 3 cols) : [1. 1. 1.]


---
## ✂️ Step 4 — Train / Test Split

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'✅ Train/Test Split (80/20 stratified)')
print(f'   Train : {X_train.shape[0]} samples')
print(f'   Test  : {X_test.shape[0]} samples')

✅ Train/Test Split (80/20 stratified)
   Train : 79 samples
   Test  : 20 samples


---
## 🌲 Step 5 — RandomForest Training with GridSearchCV

In [29]:
# ── Baseline RandomForest ──────────────────────────────────────────────────
baseline_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
baseline_rf.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline_rf.predict(X_test))
print(f'📊 Baseline RandomForest Test Accuracy : {baseline_acc * 100:.2f}%')

📊 Baseline RandomForest Test Accuracy : 100.00%


In [30]:
# ── 5-Fold Stratified Cross-Validation ────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    baseline_rf, X_scaled, y, cv=cv, scoring='accuracy', n_jobs=-1
)

print(f'📊 5-Fold Cross-Validation Results:')
print(f'   Fold Scores : {[round(s * 100, 2) for s in cv_scores]}')
print(f'   Mean        : {cv_scores.mean() * 100:.2f}%')
print(f'   Std Dev     : ± {cv_scores.std() * 100:.2f}%')

📊 5-Fold Cross-Validation Results:
   Fold Scores : [np.float64(100.0), np.float64(100.0), np.float64(95.0), np.float64(95.0), np.float64(100.0)]
   Mean        : 98.00%
   Std Dev     : ± 2.45%


In [31]:
# ── GridSearchCV — Hyperparameter Tuning ───────────────────────────────────
print('🔍 Running GridSearchCV — please wait...')

param_grid = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features'     : ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\n✅ GridSearchCV complete')
print(f'   Best Parameters : {grid_search.best_params_}')
print(f'   Best CV Score   : {grid_search.best_score_ * 100:.2f}%')

🔍 Running GridSearchCV — please wait...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

✅ GridSearchCV complete
   Best Parameters : {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
   Best CV Score   : 100.00%


In [32]:
# ── Retrain best model on FULL dataset for production ─────────────────────
best_params = grid_search.best_params_

final_model = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
final_model.fit(X_scaled, y)    # 100% of data

print(f'✅ Final model retrained on full dataset (100% samples)')
print(f'   Parameters : {best_params}')

✅ Final model retrained on full dataset (100% samples)
   Parameters : {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}


---
## 📈 Step 6 — Model Evaluation

In [33]:
y_pred   = grid_search.best_estimator_.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print('=' * 55)
print(f'  TEST SET ACCURACY : {test_acc * 100:.2f}%')
print('=' * 55)
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

  TEST SET ACCURACY : 100.00%

📋 Classification Report:
              precision    recall  f1-score   support

    10-26-26       1.00      1.00      1.00         1
    14-35-14       1.00      1.00      1.00         3
    17-17-17       1.00      1.00      1.00         1
       20-20       1.00      1.00      1.00         3
       28-28       1.00      1.00      1.00         3
         DAP       1.00      1.00      1.00         4
        Urea       1.00      1.00      1.00         5

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [34]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(10, 8))
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_encoder.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=30)
ax.set_title('Confusion Matrix — Fertilizer Prediction', fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'fertilizer_confusion_matrix.png'), dpi=100)
plt.show()
print('✅ Confusion matrix saved')

✅ Confusion matrix saved


In [35]:
# Feature Importance
importances = grid_search.best_estimator_.feature_importances_
feat_imp    = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors  = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_imp)))
feat_imp.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.set_title('Feature Importance — RandomForest Fertilizer Model',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Gini Importance Score')
for bar, val in zip(ax.patches, feat_imp):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'fertilizer_feature_importance.png'), dpi=100)
plt.show()
print('✅ Feature importance saved')

✅ Feature importance saved


In [36]:
# Performance summary
summary = pd.DataFrame({
    'Metric': [
        'Baseline RF Accuracy',
        '5-Fold CV Mean Accuracy',
        '5-Fold CV Std Dev',
        'Best GridSearch CV Score',
        'Final Test Set Accuracy'
    ],
    'Value': [
        f'{baseline_acc * 100:.2f}%',
        f'{cv_scores.mean() * 100:.2f}%',
        f'± {cv_scores.std() * 100:.2f}%',
        f'{grid_search.best_score_ * 100:.2f}%',
        f'{test_acc * 100:.2f}%'
    ]
})
print(summary.to_string(index=False))

                  Metric   Value
    Baseline RF Accuracy 100.00%
 5-Fold CV Mean Accuracy  98.00%
       5-Fold CV Std Dev ± 2.45%
Best GridSearch CV Score 100.00%
 Final Test Set Accuracy 100.00%


---
## 💾 Step 7 — Save Model Artifacts

In [37]:
MODEL_PATH          = os.path.join(MODELS_DIR, 'fertilizer_model.pkl')
SCALER_PATH         = os.path.join(MODELS_DIR, 'fertilizer_scaler.pkl')
ENCODERS_PATH       = os.path.join(MODELS_DIR, 'fertilizer_label_encoders.pkl')
TARGET_ENC_PATH     = os.path.join(MODELS_DIR, 'fertilizer_target_encoder.pkl')
METADATA_PATH       = os.path.join(MODELS_DIR, 'fertilizer_model_metadata.pkl')

joblib.dump(final_model,    MODEL_PATH);       print(f'✅ fertilizer_model.pkl          → saved')
joblib.dump(scaler,         SCALER_PATH);      print(f'✅ fertilizer_scaler.pkl         → saved')
joblib.dump(label_encoders, ENCODERS_PATH);    print(f'✅ fertilizer_label_encoders.pkl → saved')
joblib.dump(target_encoder, TARGET_ENC_PATH);  print(f'✅ fertilizer_target_encoder.pkl → saved')

metadata = {
    'feature_columns'    : FEATURE_COLS,
    'numeric_columns'    : NUM_COLS,
    'categorical_columns': CAT_COLS,
    'target_column'      : TARGET_COL,
    'target_classes'     : list(target_encoder.classes_),
    'best_params'        : best_params,
    'test_accuracy_pct'  : round(test_acc * 100, 2),
    'cv_mean_accuracy_pct': round(cv_scores.mean() * 100, 2),
}
joblib.dump(metadata, METADATA_PATH);          print(f'✅ fertilizer_model_metadata.pkl → saved')

print('\n📁 Final models/ folder:')
for f in sorted(os.listdir(MODELS_DIR)):
    size_kb = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1024
    print(f'   {f:<45} {size_kb:>7.1f} KB')

✅ fertilizer_model.pkl          → saved
✅ fertilizer_scaler.pkl         → saved
✅ fertilizer_label_encoders.pkl → saved
✅ fertilizer_target_encoder.pkl → saved
✅ fertilizer_model_metadata.pkl → saved

📁 Final models/ folder:
   crop_model.pkl                                 2625.9 KB
   fertilizer_class_dist.png                        22.7 KB
   fertilizer_confusion_matrix.png                  33.6 KB
   fertilizer_correlation.png                       35.0 KB
   fertilizer_feature_importance.png                26.4 KB
   fertilizer_label_encoders.pkl                     0.8 KB
   fertilizer_model.pkl                            584.3 KB
   fertilizer_model_metadata.pkl                     0.6 KB
   fertilizer_numeric_dist.png                      57.1 KB
   fertilizer_scaler.pkl                             0.8 KB
   fertilizer_target_encoder.pkl                     0.5 KB
   label_encoders.pkl                                0.7 KB
   scaler.pkl                                        0.

---
## 🧪 Step 8 — Inference Sanity Check

In [38]:
# Reload from disk
loaded_model   = joblib.load(MODEL_PATH)
loaded_scaler  = joblib.load(SCALER_PATH)
loaded_enc     = joblib.load(ENCODERS_PATH)
loaded_target  = joblib.load(TARGET_ENC_PATH)
loaded_meta    = joblib.load(METADATA_PATH)
print('✅ All artifacts reloaded from disk')
print(f'   Features     : {loaded_meta["feature_columns"]}')
print(f'   Target labels: {loaded_meta["target_classes"]}')

✅ All artifacts reloaded from disk
   Features     : ['Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type', 'Nitrogen', 'Potassium', 'Phosphorous']
   Target labels: ['10-26-26', '14-35-14', '17-17-17', '20-20', '28-28', 'DAP', 'Urea']


In [39]:
def predict_fertilizer(input_dict: dict):
    """
    Predict fertilizer name from raw input values.

    Parameters
    ----------
    input_dict : dict  — keys must match loaded_meta['feature_columns']

    Returns
    -------
    (str, float)  — (predicted fertilizer name, confidence %)
    """
    row = []
    for col in loaded_meta['feature_columns']:
        val = input_dict[col]
        if col in loaded_meta['categorical_columns']:
            val = loaded_enc[col].transform([str(val).strip()])[0]
        row.append(float(val))

    arr        = np.array([row], dtype=np.float64)
    arr_scaled = loaded_scaler.transform(arr)
    pred_enc   = loaded_model.predict(arr_scaled)[0]
    pred_label = loaded_target.inverse_transform([pred_enc])[0]
    confidence = round(max(loaded_model.predict_proba(arr_scaled)[0]) * 100, 2)
    return pred_label, confidence


# Test with first row of original dataset
sample_row   = df.iloc[0].to_dict()
actual_label = sample_row.pop(TARGET_COL)

predicted, conf = predict_fertilizer(sample_row)

print('🧪 Inference Sanity Check')
print(f'   Input              : {sample_row}')
print(f'   Actual Fertilizer  : {actual_label}')
print(f'   Predicted          : {predicted}')
print(f'   Confidence         : {conf}%')
print(f'   Result             : {"✅ CORRECT" if str(predicted) == str(actual_label) else "❌ MISMATCH"}')

🧪 Inference Sanity Check
   Input              : {'Temparature': 26, 'Humidity': 52, 'Moisture': 38, 'Soil Type': 'Sandy', 'Crop Type': 'Maize', 'Nitrogen': 37, 'Potassium': 0, 'Phosphorous': 0}
   Actual Fertilizer  : Urea
   Predicted          : Urea
   Confidence         : 96.1%
   Result             : ✅ CORRECT


In [40]:
print('=' * 55)
print('  🌿 FERTILIZER MODEL TRAINING COMPLETE')
print('=' * 55)
print(f'  Algorithm    : RandomForestClassifier')
print(f'  Best Params  : {best_params}')
print(f'  CV Accuracy  : {cv_scores.mean() * 100:.2f}% ± {cv_scores.std() * 100:.2f}%')
print(f'  Test Accuracy: {test_acc * 100:.2f}%')
print(f'  Output Model : fertilizer_model.pkl')
print('=' * 55)

  🌿 FERTILIZER MODEL TRAINING COMPLETE
  Algorithm    : RandomForestClassifier
  Best Params  : {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
  CV Accuracy  : 98.00% ± 2.45%
  Test Accuracy: 100.00%
  Output Model : fertilizer_model.pkl
